# Limit Comparisons: time vs NDCG@10 (A100, flan-t5-xl)

This notebook plots NDCG@10 against estimated time using the A100 latency for flan-t5-xl.
Each line stops at its peak NDCG@10, marked with an "x".

Make sure you have run:
`python scripts/refine_limit_comparisons.py`


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_CSV = Path("reports/limit_comparisons_experiment.csv")
TIME_COL = "estimated_ms_a100_flan_t5_xl"
DATASET_FILTER = None  # Example: ["dl-2019", "dl-2020"]
AGGREGATE = True  # Average across datasets
TIME_UNIT = "s"  # "ms" or "s"

df = pd.read_csv(INPUT_CSV)
if TIME_COL not in df.columns:
    raise ValueError(
        f"Missing {TIME_COL}. Run scripts/refine_limit_comparisons.py first."
    )

if DATASET_FILTER:
    df = df[df["Dataset"].isin(DATASET_FILTER)].copy()

if AGGREGATE:
    df = (
        df.groupby(["DisplayRanker", "Budget"], as_index=False)[
            ["NDCG@10", TIME_COL]
        ]
        .mean()
        .rename(columns={TIME_COL: "time_ms"})
    )
    df["label"] = df["DisplayRanker"]
else:
    df = df.rename(columns={TIME_COL: "time_ms"})
    df["label"] = df["DisplayRanker"] + " | " + df["Dataset"].astype(str)

if TIME_UNIT == "s":
    df["time"] = df["time_ms"] / 1000.0
    x_label = "Estimated time per task (s) [A100, flan-t5-xl]"
else:
    df["time"] = df["time_ms"]
    x_label = "Estimated time per task (ms) [A100, flan-t5-xl]"


In [ ]:
sns.set_style("whitegrid")

labels = sorted(df["label"].unique())
palette = sns.color_palette("tab10", n_colors=len(labels))

fig, ax = plt.subplots(figsize=(11, 6))

for label, color in zip(labels, palette):
    group = df[df["label"] == label].sort_values("time").reset_index(drop=True)
    if group.empty:
        continue

    peak_pos = int(group["NDCG@10"].idxmax())
    group_to_peak = group.iloc[: peak_pos + 1]
    peak_row = group.iloc[peak_pos]

    ax.plot(group_to_peak["time"], group_to_peak["NDCG@10"], label=label, color=color)
    ax.scatter([peak_row["time"]], [peak_row["NDCG@10"]], marker="x", s=80, color=color, zorder=3)

ax.set_xlabel(x_label)
ax.set_ylabel("NDCG@10")
ax.set_title("Limit Comparisons: NDCG@10 vs time (A100, flan-t5-xl)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
